# sLTN — Multi-Digit MNIST Addition

Two $N$-digit numbers are given as sequences of MNIST images. We observe only the digits of their sum and the final carry, never the label of any single image. From this, a CNN must learn to recognise the digits.

The example shows:

1. a structural dimension `P` (digit position), with selections such as `x1[p][p=0]`;
2. a carry chain written once as a family of macros (`sig.define`) and unrolled over the positions;
3. categorical symbols grounded as probability vectors, with addition computed exactly by table contraction.

## 1  Setup

`N` is the number of digits per operand. `P` and `P_struct` are the $p$-mean exponents of the first-order and structural quantifiers (§4).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
import torchvision
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from tqdm.notebook import tqdm

import sltn
from sltn import Signature, Interpretation, Logic, kb_describe, kb_backward
import sltn.interpretation.operators as fuzzy_ops

In [ ]:
N = 16
num_train_samples = 7500  # chosen freely, independent of N
num_test_samples = 1500
STABLE = True
EPS = 1e-12
P = 4
P_struct = 2
batch_size = 16
nr_of_epochs = 10
learning_rate = 1e-4

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

## 2  Signature

| Symbol | Kind | Declaration | Meaning |
|---|---|---|---|
| `Image`, `Digit`, `Carry` | sorts | | an MNIST image; a distribution over 0–9; a distribution over {0, 1} |
| `P` | dimension | | digit positions, 0 = least significant |
| `p` | structural variable | over `P` | a position |
| `x1`, `x2` | variables | `Image`, dims `P` | the two operands |
| `s` | variable | `Digit`, dims `P` | the digits of the sum (observed) |
| `c_true` | variable | `Carry` | the final carry (observed) |
| `digit` | function | `Image → Digit` | the CNN to be learned |
| `sumDigit`, `carryOut` | functions | `Digit × Digit × Carry → Digit` / `Carry` | one step of long addition |
| `=d`, `=c` | infix predicates | | agreement of two distributions |
| `zero_carry`, `d0`…`d9` | constants | `Carry`, `Digit` | the zero carry; the ten digit classes |

All functions act on a single position: applied to `x1[p]` they are applied position by position, and `[p=k]` selects position `k`.

In [ ]:
sig = Signature("Mnist_sum")
sig.sort("Image")
sig.sort("Digit")
sig.sort("Carry")
sig.dimension("P")
sig.structural_variable("p", "P")
sig.variable("x1", "Image", dims=["P"])
sig.variable("x2", "Image", dims=["P"])
sig.variable("s",  "Digit", dims=["P"])
sig.variable("c_true", "Carry")   # ground-truth final carry-out for the whole example
sig.function("digit", ["Image"], "Digit")
sig.function("sumDigit", ["Digit", "Digit", "Carry"], "Digit")
sig.function("carryOut", ["Digit", "Digit", "Carry"], "Carry")
sig.predicate("=d", ["Digit", "Digit"], infix=True)
sig.predicate("=c", ["Carry", "Carry"], infix=True)
sig.constant("zero_carry", "Carry")

# digit constants
for i in range(10):
    sig.constant(f"d{i}", "Digit")

sig

## 3  Knowledge base

**Carry chain.** The carry leaving position $k$ is a macro, defined recursively and expanded at parse time:

$$
c_0 := \mathrm{carryOut}(\mathrm{digit}(x_1[p])[p{=}0],\ \mathrm{digit}(x_2[p])[p{=}0],\ \mathrm{zero\_carry}),
\qquad
c_k := \mathrm{carryOut}(\mathrm{digit}(x_1[p])[p{=}k],\ \mathrm{digit}(x_2[p])[p{=}k],\ c_{k-1}).
$$

**Clauses.** With $c_{-1} := \mathrm{zero\_carry}$:

| Clause | Formula | Role |
|---|---|---|
| $\phi_{s,k}$, $k = 0..N{-}1$ | $\forall (x_1, x_2, s): s[p][p{=}k] =_d \mathrm{sumDigit}(\mathrm{digit}(x_1[p])[p{=}k],\ \mathrm{digit}(x_2[p])[p{=}k],\ c_{k-1})$ | each sum digit is observed |
| $\phi_c$ | $\forall (x_1, x_2, c_\mathrm{true}): c_{N-1} =_c c_\mathrm{true}$ | the final carry is observed |
| $\kappa_i$, $i = 0..9$ | $\exists x_1: \exists p: \mathrm{digit}(x_1[p]) =_d d_i$ | every class is used: prevents collapsing several digits onto one class |

The clauses are unrolled over positions because the carry chain is sequential: position $k$ depends on all positions before it. Gradients flow through the expanded macros into the CNN.

The cells below define the class-use clauses $\kappa_i$ first, then the carry macros and the addition clauses.

In [ ]:
# Class-use clauses: every digit class occurs somewhere in the batch.
class_use = [sig.parse(f"∃ x1: ∃ p: digit(x1[p]) =d d{i}") for i in range(10)]
class_use

In [ ]:
for p in range(N):
    if p == 0:
        body = "carryOut(digit(x1[p])[p=0], digit(x2[p])[p=0], zero_carry)"
    else:
        body = f"carryOut(digit(x1[p])[p={p}], digit(x2[p])[p={p}], c{p-1}(x1,x2))"
    sig.define(f"c{p}", ["x1", "x2"], body)

phi_s0 = "∀ (x1,x2,s) : (s[p][p=0] =d sumDigit(digit(x1[p])[p=0], digit(x2[p])[p=0], zero_carry))"
phi_sp = [
    f"∀ (x1,x2,s) : (s[p][p={p}] =d sumDigit(digit(x1[p])[p={p}], digit(x2[p])[p={p}], c{p-1}(x1,x2)))"
    for p in range(1, N)
]
phi_cN = f"∀ (x1,x2,c_true) : (c{N-1}(x1,x2) =c c_true)"


kb = sltn.KB([
    sig.parse(phi_s0),
    *[sig.parse(f) for f in phi_sp],
    sig.parse(phi_cN),
    *class_use
    ])

kb_dict = kb_describe(kb)

## 4  Semantics

**Logic.** Product conjunction, $p$-mean quantifiers (`P` for first-order, `P_struct` for structural) and `AggregPMeanError` as `SatAgg`. `with_defaults` derives `Or`, `Implies` and `Equiv` from `And` and `Not`.

In [ ]:
logic = Logic(
    And=fuzzy_ops.AndProd(stable=STABLE, eps=EPS),
    Not=fuzzy_ops.NotStandard(),
).with_defaults(stable=STABLE, eps=EPS)
logic["∃"] = fuzzy_ops.AggregPMean(stable=STABLE, p=P, eps=EPS)
logic["∀"] = fuzzy_ops.AggregPMean(stable=STABLE, p=1/P, eps=EPS)
logic["SatAgg"] = fuzzy_ops.AggregPMeanError(stable=STABLE, p=P, eps=EPS)
logic["∀,p"] = fuzzy_ops.AggregPMean(stable=STABLE, p=1/P_struct, eps=EPS)
logic["∃,p"] = fuzzy_ops.AggregPMean(stable=STABLE, p=P_struct, eps=EPS)

print(logic)

**Groundings.**

- `digit` is a small CNN with a softmax output: a full probability vector, never collapsed to a scalar.
- `sumDigit` and `carryOut` contract fixed tables with `torch.einsum`: `sum_table[i, j, k, s] = 1` iff `(i + j + k) % 10 == s`, and likewise for the carry with `// 10`. For independent input distributions this gives the exact output distribution, so no smooth approximation of `mod` or `div` is needed.
- `=d` and `=c` are dot products: the probability that the prediction assigns to the target class. Its gradient through the softmax behaves like cross-entropy's (`pred - target`), so it stays informative when the prediction is uncertain.

In [ ]:
class DigitCNN(torch.nn.Module):
    """CNN grounding `digit`: Image -> full probability vector over classes 0..9."""
    def __init__(self):
        super().__init__()
        self.conv1 = torch.nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = torch.nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.fc1   = torch.nn.Linear(32 * 7 * 7, 128)
        self.fc2   = torch.nn.Linear(128, 10)

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.max_pool2d(x, 2)
        x = torch.relu(self.conv2(x))
        x = torch.max_pool2d(x, 2)
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        logits = self.fc2(x)
        return torch.softmax(logits, dim=-1)


cnn = DigitCNN()

# Exact addition tables: sum_table[i,j,k,s] = 1 iff (i+j+k)%10==s; carry_table analogous.
sum_table = torch.zeros(10, 10, 2, 10)
carry_table = torch.zeros(10, 10, 2, 2)
for i in range(10):
    for j in range(10):
        for k in range(2):
            total = i + j + k
            sum_table[i, j, k, total % 10] = 1.0
            carry_table[i, j, k, total // 10] = 1.0

In [ ]:
interp = Interpretation(sig).to(device=device)
interp["Image"] = sltn.Type("Image", shape=(1, 28, 28), axis_names=["channel", "height", "width"])
interp["Digit"] = sltn.Type("Digit", shape=(10,), axis_names=["digit_class"])
interp["Carry"] = sltn.Type("Carry", shape=(2,), axis_names=["carry_class"])

interp["digit"] = cnn.to(device=device)

_sum_table = sum_table.to(device=device)
_carry_table = carry_table.to(device=device)
interp["sumDigit"] = lambda d1, d2, cin: torch.einsum("...i,...j,...k,ijks->...s", d1, d2, cin, _sum_table)
interp["carryOut"] = lambda d1, d2, cin: torch.einsum("...i,...j,...k,ijkc->...c", d1, d2, cin, _carry_table)

# Probability mass on the true class, clamped to guard against float rounding past [0, 1].
interp["=d"] = lambda a, b: (a * b).sum(dim=-1, keepdim=True).clamp(0.0, 1.0)
interp["=c"] = lambda a, b: (a * b).sum(dim=-1, keepdim=True).clamp(0.0, 1.0)

interp["zero_carry"] = torch.tensor([1.0, 0.0], device=device) # zero carry class

for i in range(10):
    interp[f"d{i}"] = torch.zeros(10, device=device)
    interp[f"d{i}"][i] = 1.0

## 5  Data

`generate_sums(n, dataset, num_samples)` draws every digit image uniformly with replacement and computes the sum digits and carries by long addition. The number of samples is therefore independent of `n`. Calling it at every epoch (§6) produces new pairings; this matters because the knowledge base constrains images only through their sums, never individually. Pixel values are scaled to [0, 1]. The digit labels `d1`, `d2` are kept for evaluation only.

In [ ]:
mnist_train = torchvision.datasets.MNIST(
    "../datasets/", train=True, download=True,
    transform=torchvision.transforms.ToTensor())

mnist_test = torchvision.datasets.MNIST(
    "../datasets/", train=False, download=True,
    transform=torchvision.transforms.ToTensor())

def generate_sums(n, dataset, num_samples):
    """Build `num_samples` random n-digit addition examples. Every digit image is drawn
    uniformly with replacement, so each call produces new pairings."""
    data_shape = dataset.data[0].shape
    idx1 = torch.randint(0, len(dataset.data), (num_samples, n))
    idx2 = torch.randint(0, len(dataset.data), (num_samples, n))
    data1 = dataset.data[idx1].reshape((num_samples, n) + data_shape).to(torch.float) / 255
    data2 = dataset.data[idx2].reshape((num_samples, n) + data_shape).to(torch.float) / 255
    digit1 = dataset.targets[idx1]
    digit2 = dataset.targets[idx2]
    sums = torch.zeros((num_samples,n),dtype=torch.int)
    carries = torch.zeros((num_samples,n),dtype=torch.int)
    sums[:,0] = (digit1[:,0] + digit2[:,0]) % 10
    carries[:,0] = (digit1[:,0] + digit2[:,0] ) // 10
    for i in range(1,n):
        sums[:,i] = (digit1[:,i] + digit2[:,i] + carries[:,i-1]) % 10
        carries[:,i] = (digit1[:,i] + digit2[:,i] + carries[:,i-1]) // 10
    return {"x1":data1, "x2":data2, "d1":digit1, "d2":digit2, "s":sums, "c":carries}

train_dataset = generate_sums(N, mnist_train, num_samples=num_train_samples)
test_dataset = generate_sums(N, mnist_test, num_samples=num_test_samples)

print(train_dataset["x1"].shape)
print(train_dataset["d1"][0], train_dataset["d2"][0], train_dataset["s"][0], train_dataset["c"][0])
print(test_dataset["d1"][0], test_dataset["d2"][0], test_dataset["s"][0], test_dataset["c"][0])

In [ ]:
class DataLoader:
    def __init__(self, dataset, batch_size=32, shuffle=True):
        self.dataset = dataset
        self.dataset_len = dataset["d1"].shape[0]
        self.batch_size = batch_size
        self.shuffle = shuffle

    def __len__(self):
        return int(np.ceil(self.dataset_len / self.batch_size))

    def __iter__(self):
        idxlist = list(range(self.dataset_len))
        if self.shuffle:
            np.random.shuffle(idxlist)
        for start in range(0, self.dataset_len, self.batch_size):
            idx = idxlist[start:start + self.batch_size]
            yield {k:self.dataset[k][idx] for k in self.dataset} | {"idx":idx}

train_dataloader = DataLoader(train_dataset,batch_size=batch_size,shuffle=True)
test_dataloader = DataLoader(test_dataset,batch_size=batch_size,shuffle=False)


def plot_sample(samples, idx):
    pics1 = samples["x1"][idx]
    pics2 = samples["x2"][idx]
    fig = plt.figure(figsize=(N,2))
    for i in range(N):
        plt.subplot(2,N,i+1)
        plt.imshow(pics1[i].cpu().squeeze().numpy(),cmap='gray')
        plt.axis('off')
        plt.title(f"{samples['d1'][idx][i]}")
    for i in range(N):
        plt.subplot(2,N,N+i+1)
        plt.imshow(pics2[i].cpu().squeeze().numpy(),cmap='gray')
        plt.axis('off')
        plt.title(f"{samples['d2'][idx][i]}")
    plt.tight_layout()
    plt.show()
    print("sums:", [int(i) for i in samples["s"][idx]])
    print("carries:", [int(i) for i in samples["c"][idx]])

for samples in train_dataloader:
    plot_sample(samples,0)
    break

## 6  Training

Each epoch draws new training pairs. Each step binds `x1`, `x2`, the one-hot sum digits `s` and the one-hot final carry `c_true`, and calls `kb_backward`. The intermediate carries need no binding: they are the expanded macros. After each epoch, we print the digit accuracy on train and test (the test set is fixed) and the satisfaction of each clause.

In [ ]:
def compute_metrics(interp, dataloader):
    acc = 0
    total = 0
    for samples in tqdm(dataloader):
        interp["x1"] = samples["x1"].unsqueeze(-3).to(device=device)
        interp["x2"] = samples["x2"].unsqueeze(-3).to(device=device)
        preds_d1 = torch.argmax(interp("digit(x1[p])").value, dim=-1).cpu()
        preds_d2 = torch.argmax(interp("digit(x2[p])").value, dim=-1).cpu()
        acc += (preds_d1.flatten() == samples["d1"].flatten()).sum()
        acc += (preds_d2.flatten() == samples["d2"].flatten()).sum()
        total += preds_d1.numel() + preds_d2.numel()
    return acc / total

In [ ]:
optimizer = torch.optim.Adam(interp["digit"].parameters(), lr=learning_rate)

for epoch in range(nr_of_epochs):
    train_dataset = generate_sums(N, mnist_train, num_samples=num_train_samples)
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    train_loss = 0.0
    interp["digit"].train()
    interp.logic = logic
    for samples in tqdm(train_dataloader):
        interp["x1"] = samples["x1"].unsqueeze(-3).to(device=device)
        interp["x2"] = samples["x2"].unsqueeze(-3).to(device=device)
        interp["s"] = F.one_hot(samples["s"].long(), num_classes=10).float().to(device=device)
        interp["c_true"] = F.one_hot(samples["c"][:, -1].long(), num_classes=2).float().to(device=device)

        optimizer.zero_grad()
        stats = kb_backward(optimizer, interp, kb_dict, aggregator="satagg")
        optimizer.step()

    with torch.no_grad():
        train_acc = compute_metrics(interp, train_dataloader)
        test_acc = compute_metrics(interp, test_dataloader)
    print("train_acc, test_acc:", train_acc, test_acc)
    losses_str = " | ".join(
        f"{name}={sat:.3f}" for name, sat in stats["sats"].items()
    )
    w_str = " ".join(f"{w:.2f}" for w in (stats["weights"] or []))
    print(f"epoch {epoch:4d} | [{losses_str}] | α=[{w_str}]")

## 7  Evaluation

`compute_metrics` compares `argmax digit(x)` with the MNIST labels, which are never used in training. Below, the same comparison as a per-class report and confusion matrix on the test set.

In [ ]:
# ── Test-set classification report + confusion matrix for `digit` ─────────
interp["digit"].eval()
digit_names = [str(d) for d in range(10)]

test_preds, test_labels = [], []
with torch.no_grad():
    for samples in test_dataloader:
        interp["x1"] = samples["x1"].unsqueeze(-3).to(device=device)
        interp["x2"] = samples["x2"].unsqueeze(-3).to(device=device)
        preds_d1 = torch.argmax(interp("digit(x1[p])").value, dim=-1).cpu().flatten()
        preds_d2 = torch.argmax(interp("digit(x2[p])").value, dim=-1).cpu().flatten()

        test_preds.extend(preds_d1.tolist() + preds_d2.tolist())
        test_labels.extend(samples["d1"].flatten().tolist() + samples["d2"].flatten().tolist())

print(classification_report(test_labels, test_preds, target_names=digit_names))

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    test_labels, test_preds, display_labels=digit_names, ax=ax, cmap="Blues",
)
ax.set_title("Digit-recognition confusion matrix (test set)")
plt.tight_layout(); plt.show()